# AI CV Reviewer: run in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarAfet/sdaia-agentic-ai-cv-reviewer/blob/main/notebooks/run_in_colab.ipynb)

Run the cells in order. Steps 1–4 need no API key. Steps 5–9 call the Groq API.

**Before you start:** get a free Groq key at <https://console.groq.com/keys>. In Colab, open the
key icon (Secrets) in the left panel, add a secret named `GROQ_API_KEY`, and turn on
"Notebook access". If you skip this, step 3 asks for the key.

## 1. Get the code and install it

In [ ]:
import os, sys, subprocess

REPO = "https://github.com/OmarAfet/sdaia-agentic-ai-cv-reviewer.git"
ROOT = "/content/sdaia-agentic-ai-cv-reviewer"

if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "--depth", "1", REPO, ROOT], check=True)
os.chdir(ROOT)

# Editable install: config.py finds data/, logs/ and reports/ relative to the source folder.
!pip install -q -e . pytest fpdf2

# The running kernel does not read the new editable-install path; add it directly.
sys.path.insert(0, f"{ROOT}/src")
print(sys.version)
assert sys.version_info >= (3, 12), "The project needs Python 3.12 or later."

## 2. Run the offline tests (no API key, about 1 s)

In [ ]:
!python -m pytest -q

## 3. Set the Groq API key

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("Key loaded from Colab Secrets.")
except Exception:
    from getpass import getpass
    os.environ["GROQ_API_KEY"] = getpass("GROQ_API_KEY: ")

## 4. List the jobs and the sample CVs

In [ ]:
!cv-review jobs
!ls data/cvs

## 5. Review one CV (automatic approval)

`auto_approver` accepts the AI proposal. Use it only for demos; step 6 shows the human approval.

In [ ]:
from IPython.display import Markdown, display
from cv_reviewer.pipeline import review_cv, auto_approver

out = review_cv("data/cvs/strong_backend.pdf", "backend-engineer", approver=auto_approver)
print("status:", out["state"]["status"])
display(Markdown(out["state"]["report_markdown"]))

## 6. Review one CV with human approval

The workflow pauses and asks you for a decision: `shortlist`, `reject` or `hold`.
This sample has hidden white text that tries to trick the AI, so it goes to quarantine.

In [ ]:
from cv_reviewer.cli import interactive_approver

out = review_cv("data/cvs/injection_hidden_text.pdf", "backend-engineer",
                approver=interactive_approver)
display(Markdown(out["state"]["report_markdown"]))

## 7. Review your own CV (PDF, DOCX or TXT)

In [ ]:
from google.colab import files

uploaded = files.upload()
for name, data in uploaded.items():
    path = f"/content/{name}"
    with open(path, "wb") as f:
        f.write(data)
    out = review_cv(path, "backend-engineer", approver=interactive_approver)
    display(Markdown(out["state"]["report_markdown"]))

## 8. Run all 12 sample cases

Free-tier Groq limits make this take a few minutes. The client waits on HTTP 429.

In [ ]:
!python scripts/run_demo.py

## 9. Look at the trace of the last run

In [ ]:
import json
run_id = out["state"]["run_id"]
with open(f"logs/{run_id}.jsonl") as f:
    for line in f:
        e = json.loads(line)
        print(e.get("event"), {k: v for k, v in e.items() if k not in ("event", "ts", "run_id")})
print(json.dumps(out["summary"], indent=2))

## 10. Download the reports and logs

In [ ]:
!zip -qr /content/cv_reviewer_output.zip reports logs
files.download("/content/cv_reviewer_output.zip")